# Sample 1 — Tokenization by hand

Before a transformer can do anything with text, the text has to become numbers. This notebook
builds that step from scratch: no libraries, just Python, so every rule is visible.

**Goals**
1. Split a tiny corpus into tokens and build a vocabulary.
2. Map tokens to integer ids and back.
3. Compare a word-level vocabulary to a character-level one.
4. See why real-world tokenizers (BPE, used by GPT-2) exist — a preview for `sample-7`.


## 1. A tiny corpus

We'll use one short sentence as our entire "training corpus." In a real system this would be
gigabytes of text; here it's small enough that you can build the vocabulary by hand and check
every step.

In [1]:
corpus = "the cat sat on the mat the cat likes the mat"
print(corpus)


the cat sat on the mat the cat likes the mat


## 2. Word-level vocabulary

Split on whitespace, collect the *unique* words, and assign each one an integer id. The order
doesn't matter for correctness — we just need it to be consistent between encode and decode.

In [2]:
words = corpus.split()
unique_words = sorted(set(words))

token_to_id = {token: idx for idx, token in enumerate(unique_words)}
id_to_token = {idx: token for token, idx in token_to_id.items()}

print(f"{len(words)} tokens in the corpus, {len(unique_words)} unique words")
token_to_id


11 tokens in the corpus, 6 unique words


{'cat': 0, 'likes': 1, 'mat': 2, 'on': 3, 'sat': 4, 'the': 5}

## 3. Encode / decode

`encode` turns a string into a list of ids using the vocabulary we just built. `decode` reverses
it. Round-tripping a sentence should return exactly the original text (for words already in the
vocabulary).

In [3]:
def encode(text, vocab):
    return [vocab[word] for word in text.split()]

def decode(ids, id_to_token):
    return " ".join(id_to_token[i] for i in ids)

sentence = "the cat sat on the mat"
ids = encode(sentence, token_to_id)
print("ids:   ", ids)
print("decode:", decode(ids, id_to_token))
assert decode(ids, id_to_token) == sentence


ids:    [5, 0, 4, 3, 5, 2]
decode: the cat sat on the mat


In [4]:
# What happens with a word that isn't in the vocabulary?
try:
    encode("the dog sat", token_to_id)
except KeyError as e:
    print(f"KeyError: {e} is not in the vocabulary")


KeyError: 'dog' is not in the vocabulary


This is the **out-of-vocabulary (OOV) problem**: a word-level tokenizer can only represent
words it has seen before. "dog" never appeared in our corpus, so there's no id for it — the model
would have no way to process that sentence at all.

## 4. Character-level vocabulary

One fix: tokenize by *character* instead of by word. The vocabulary becomes tiny (just the
letters and space used), and any *new word built from those same letters* can still be spelled
out — but sequences get much longer.

In [5]:
chars = sorted(set(corpus))
char_to_id = {ch: idx for idx, ch in enumerate(chars)}
id_to_char = {idx: ch for ch, idx in char_to_id.items()}

print(f"Character vocabulary size: {len(chars)}")
print(char_to_id)

# "sail" never appeared as a word in the corpus, but every letter in it did —
# so the character-level tokenizer has no trouble with it.
new_word = "the cat likes to sail"
char_ids = [char_to_id[ch] for ch in new_word]
print(f"\n'{new_word}' as char ids:", char_ids)
print("decoded:", "".join(id_to_char[i] for i in char_ids))


Character vocabulary size: 13
{' ': 0, 'a': 1, 'c': 2, 'e': 3, 'h': 4, 'i': 5, 'k': 6, 'l': 7, 'm': 8, 'n': 9, 'o': 10, 's': 11, 't': 12}

'the cat likes to sail' as char ids: [12, 4, 3, 0, 2, 1, 12, 0, 7, 5, 6, 3, 11, 0, 12, 10, 0, 11, 1, 5, 7]
decoded: the cat likes to sail


In [6]:
word_len = len(encode(sentence, token_to_id))
char_len = len([char_to_id[ch] for ch in sentence])
print(f"'{sentence}'")
print(f"  word-level tokens: {word_len}")
print(f"  char-level tokens: {char_len}")


'the cat sat on the mat'
  word-level tokens: 6
  char-level tokens: 22


## 5. The trade-off, and what real tokenizers do

- **Word-level**: short sequences, but a huge vocabulary and the OOV problem — any unseen word
  breaks it.
- **Character-level**: no OOV problem and a tiny vocabulary, but sequences get long, and the
  model has to work harder to learn that `"c"`, `"a"`, `"t"` in sequence means something.

Real tokenizers (GPT-2's BPE, WordPiece, SentencePiece, ...) sit in between: they learn a
vocabulary of common *subword* pieces from a large corpus, so common words get one token, rare
words get split into a few meaningful chunks, and there's no OOV problem. We'll load GPT-2's real
tokenizer and see this directly in `sample-7-pretrained-inference-huggingface`.

**Next:** `sample-2-embeddings` turns these integer ids into vectors.